# PlantSeg U-Net Training Workflow

Step-by-step notebook for binary disease-mask segmentation with U-Net, focal loss, weighted BCE, Dice loss, MLflow, and TensorBoard.

Use the CUDA environment only: `F:\\PyTorch_GPU\\torch_gpu\\Scripts\\python.exe`.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path(r"F:\\PyTorch_GPU\\Plant_Seg")
SRC_DIR = PROJECT_ROOT / "plantseg_training" / "src"
sys.path.insert(0, str(SRC_DIR))

CONFIG_PATH = PROJECT_ROOT / "plantseg_training" / "configs" / "unet_binary.json"
DATA_ROOT = PROJECT_ROOT / "Data_exploration" / "data" / "archive" / "plantsegv2"
print(PROJECT_ROOT)
print(CONFIG_PATH)

## 1. Verify CUDA and package versions

In [ ]:
import torch
import cv2
import numpy as np

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("cv2:", cv2.__version__)
print("numpy:", np.__version__)

## 2. Inspect dataset size and imbalance

`Mask ratio` estimates the positive-pixel rate. We use it to auto-compute `pos_weight` for weighted BCE.

In [ ]:
import pandas as pd

metadata = pd.read_csv(DATA_ROOT / "Metadatav2.csv")
display(metadata.head())
display(metadata["Split"].value_counts())

train = metadata[metadata["Split"] == "Training"].copy()
positive_ratio = train["Mask ratio"].clip(lower=1e-6, upper=1.0).mean()
pos_weight = min((1.0 - positive_ratio) / positive_ratio, 25.0)
print(f"train positive pixel ratio ~= {positive_ratio:.6f}")
print(f"auto pos_weight capped at 25 ~= {pos_weight:.4f}")
display(train["Disease"].value_counts().head(20))

## 3. Review experiment config

In [ ]:
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

## 4. Smoke-test one batch and one CUDA forward pass

In [ ]:
from plantseg_training.data import make_datasets
from plantseg_training.models import build_model

train_ds, val_ds, test_ds = make_datasets(config["dataset"])
sample = train_ds[0]
print(sample["pixel_values"].shape, sample["labels"].shape, sample["labels"].float().mean().item())

model_cfg = dict(config["model"])
model_cfg["loss"] = dict(model_cfg["loss"])
model_cfg["loss"]["pos_weight"] = float(pos_weight)
model = build_model(model_cfg).cuda()
x = sample["pixel_values"].unsqueeze(0).cuda()
y = sample["labels"].unsqueeze(0).cuda()
out = model(pixel_values=x, labels=y)
print("loss:", float(out["loss"].detach().cpu()))
print("logits:", tuple(out["logits"].shape))

## 5. Launch training

Recommended from PowerShell, so the process is easy to monitor and stop:

```powershell
F:\\PyTorch_GPU\\torch_gpu\\Scripts\\python.exe plantseg_training/train.py --config plantseg_training/configs/unet_binary.json
```

In [ ]:
# Optional notebook launch. PowerShell launch is usually cleaner for long runs.
# from plantseg_training.trainer import run_training
# run_training(config=config, config_path=CONFIG_PATH)

## 6. Live monitoring commands

Run these in separate PowerShell windows.

In [ ]:
print(r"nvidia-smi -l 5")
print(r"F:\PyTorch_GPU\torch_gpu\Scripts\python.exe -m mlflow ui --backend-store-uri plantseg_training/outputs/mlruns")
print(r"F:\PyTorch_GPU\torch_gpu\Scripts\python.exe -m tensorboard.main --logdir plantseg_training/outputs/unet_binary/runs")
print(r"Get-Content plantseg_training/outputs/unet_binary/logs/training_history.csv -Tail 20 -Wait")

## 7. Read final metrics after training

In [ ]:
metrics_path = PROJECT_ROOT / "plantseg_training" / "outputs" / "unet_binary" / "final_metrics.json"
if metrics_path.exists():
    print(metrics_path.read_text(encoding="utf-8"))
else:
    print("No final_metrics.json yet. Training has not completed.")